# Gurukul AI — Adaptive Bilingual AI Tutor for Indian Competitive Exams

*Submitted to: Kaggle × Google — Agents for Good 2026*  
*Author: Ravi Kumar Prasada (rkprasada@gmail.com)*  
*GitHub: [https://github.com/RKPrasada/GURUKULAI](https://github.com/RKPrasada/GURUKULAI)*  
*Android APK: [Download from GitHub Releases](https://github.com/RKPrasada/GURUKULAI/releases/tag/v1.0-kaggle)*

## The Problem

India runs some of the world's most competitive public examinations. The **Railway Recruitment Board (RRB)** exams receive over **2.5 crore (25 million) applications** per cycle. NDA, JEE, and NEET together see tens of millions more.

Most applicants come from towns and villages where quality coaching is either unavailable or costs ₹60,000–₹2,00,000 per year — equivalent to a family's entire annual savings.

> **The result: exam outcome is dictated more by zip code and family income than by ability.**

**Gurukul AI** is a free, bilingual (English + Hindi) AI tutor that gives every aspirant access to the kind of personalised, adaptive preparation that was previously only available to students who could afford expensive coaching centres.

## What I Built

Gurukul AI is a **multi-agent AI system** with three interfaces:

| Interface | Description |
|---|---|
| React 18 Web App | Full-featured dashboard — all features |
| Flutter Android APK | 54.8 MB, sideloadable, works with local or cloud backend |
| FastAPI Backend | 8 specialised AI agents + 11 route files |

**Exams supported (8 total):**  
RRB NTPC · RRB ALP · RRB Group D · RRB Technician · RRB JE · NDA · JEE · NEET

**Video Demo:** [https://youtu.be/AkeJM2bScQU](https://youtu.be/AkeJM2bScQU)

## Agent Architecture

The system is built around a central **OrchestratorAgent** that routes incoming student messages to the most relevant specialist agent using keyword scoring:

```
Student message
      │
      ▼
OrchestratorAgent
  (keyword scoring → AgentRegistry)
      │
      ├── DiagnosticAgent    — 3-stage adaptive placement test
      ├── ContentAgent       — study notes + curated YouTube videos
      ├── AssessmentAgent    — adaptive MCQs (difficulty from weakness_map)
      ├── FeedbackAgent      — wrong-answer explanations, bilingual
      ├── ProgressAgent      — personalised study plan generation
      ├── ScheduleAgent      — weekly/monthly calendar creation
      └── NagaAgent          — student ↔ human mentor interactions
```

**DabbuAgent** (autonomous, runs in background):
```
DabbuAgent
  ├── propose_study_plan()             → writes to study_plans/{id}_proposed.json
  ├── propose_notes()                  → writes to notes/{exam}/ (status=pending)
  ├── curate_videos()                  → YouTube search + content filter
  └── propose_progress_intervention()  → saves to dabbu/interventions.jsonl
```

### The Human-in-the-Loop: NAGA

**Dabbu** (the AI agent) never directly modifies student data. Every proposal goes through **NAGA**, a named human mentor who reviews, approves, or amends Dabbu's suggestions before students see them:

```
Dabbu proposes → NAGA reviews → Student sees (approved only)
```

NAGA has a dedicated dashboard with:
- **Approvals tab** — pending study plans, notes, video curations, progress interventions, keyword blocklist management
- **Classes tab** — schedule group sessions
- **Meetings tab** — handle student 1-to-1 meeting requests

## Key Features

### 1. Adaptive Diagnostic (3-Stage)
Students begin with a placement test. DiagnosticAgent runs three stages — broad subjects → topic drill-down → atomic concepts — building a `weakness_map` that scores every topic 0–1. All subsequent content, MCQs, and study plans are derived from this map.

### 2. SM-2 Spaced Repetition
Every topic tracks `ease_factor`, `interval_days`, `next_review_date`, and `overdue` status following the SM-2 algorithm. The Progress page shows a review schedule sorted by urgency, with overdue topics highlighted in red.

### 3. Saturday Mock Tests (Auto-Generated)
A background scheduler fires every Saturday at 15:00. `MockPaperGenerator` calls the LLM section-by-section to produce a full exam paper (75–180 questions depending on exam). Each question passes 9 validation checks. Papers are cached; if the LLM fails, the previous week's archive is used as fallback.

**Mock test UI features:** countdown timer, auto-submit at zero, autosave every 30 seconds (resume on refresh), section tabs, question navigator grid (blue=answered, orange=flagged, purple=answered+flagged), flagged-question review modal before submit, negative marking (all RRB: −1/3, JEE/NEET: −1).

### 4. 60-Day Streak Heatmap
The Progress screen shows a GitHub-style activity grid of the last 60 days, with four intensity levels based on sessions completed per day.

### 5. Triple Bar Chart (Initial / Current / Target@80%)
The overview chart shows three bars per topic: the student's score at first diagnostic, their current score, and the 80% target line — making improvement visually concrete.

### 6. Bilingual (English + Hindi)
Every screen, alert, label, and agent response is fully bilingual. Language preference is stored in the student profile and applies across both web and mobile.

### 7. Content Safety Filter
A two-tier keyword filter (BLOCKED / FLAGGED) screens all curated YouTube videos before they reach students. The blocklist is NAGA-editable from the dashboard. No LLM in the filter path — fast, deterministic, and auditable.

## Technical Stack

| Component | Technology |
|---|---|
| Backend | FastAPI (Python), SQLite, JSONL flat files |
| LLM | OpenRouter (5-model free-tier fallback chain) |
| Web frontend | React 18 + Vite + Recharts + Zustand + Tailwind CSS |
| Mobile | Flutter (Dart) → Android APK |
| Auth | HMAC JWT tokens, bcrypt hashed passwords |
| Deployment | Google Cloud Run (backend), static hosting ready (web) |

**OpenRouter model fallback chain** (tried in order, first 200 response wins):
1. `nvidia/nemotron-3-ultra-550b-a55b:free`
2. `nvidia/nemotron-3-super-120b-a12b:free`
3. `google/gemma-4-26b-a4b-it:free`
4. `openai/gpt-oss-20b:free`
5. `liquid/lfm-2.5-1.2b-instruct:free`

**Codebase scale:**
- 20 Python agent/utility modules
- 11 FastAPI route files (2,900+ lines of routes)
- 30 React/TypeScript components and pages
- 19 Flutter/Dart screens and services
- Full audit trail: every API call logged to `data/audit.jsonl` with SHA-256 hash chain

## Security Architecture

Security was designed from day one, not bolted on:

| Layer | What it does |
|---|---|
| InjectionDetector | 40+ prompt injection patterns blocked at input (EN, HI, obfuscated, base64) |
| PIIScrubber | Strips Aadhaar numbers, PAN, phone, email, voter ID before any LLM sees the text |
| InputGuard | Combines injection detection + PII scrub — applied to every free-text surface |
| OutputGuard | Blocks system prompt leakage in agent responses |
| VibeDiff | Irreversible actions (calendar create, email send, delete) need a `pending_token` confirmed in a separate `/confirm` call |
| QuarantineManager | 3 threats in 10 minutes → 30-minute block (SQLite-backed) |
| AuditLogger | Append-only JSONL with SHA-256 hash chain — tamper-evident |

**Rule: "direct agent call" is not an exemption.** Every endpoint that sends user text to an LLM — including those that bypass orchestrator routing — carries the full InputGuard → OutputGuard → AuditLogger obligation.

**Guardrail coverage — all three frontends:**

| Surface | Guard |
|---|---|
| AI Tutor chat | Orchestrator InputGuard |
| AI Tutor notes | InputGuard inline in route |
| Practice Test chat | Orchestrator InputGuard |
| Ask NAGA question form | `_input_guard` in mentor.py, HTTP 400 on threat |

Guardrail rejections are **never silent** — web shows an amber shield card, mobile shows an amber snackbar.

## How I Built This with Vibe Coding

This project was built entirely using **Claude Code** (AI-assisted coding in the terminal). The workflow:

1. Described the overall architecture and problem domain in natural language
2. Claude Code generated the initial FastAPI skeleton, agent classes, and React pages
3. Each iteration: describe a feature → Claude generates → I review for correctness, safety, and exam-domain accuracy
4. Security architecture was co-designed: described the threat model, Claude implemented the guard layers and propagated them consistently across all endpoints
5. The Flutter app was built in parallel — all 19 screens generated from descriptions of the web app's features

The AI handled the volume of boilerplate (JSONL persistence, auth middleware, chart components, Flutter state management) while I focused on **domain correctness**: verifying exam configs against official RRB PDFs, checking negative marking rules, validating syllabus coverage.

> **Key vibe coding insight**: the most valuable use of AI coding assistance is not writing CRUD — it is implementing safety and consistency patterns uniformly across a large codebase. The InputGuard/OutputGuard pattern was applied consistently to all 8 agents and 11 route files because Claude Code could propagate it everywhere at once.

## Impact & Social Good

| Dimension | Detail |
|---|---|
| Target users | Government job aspirants in Tier 2/3 Indian cities who cannot afford ₹60,000/year coaching |
| Languages | English and Hindi — covers ~90% of RRB exam aspirants |
| Reach | 8 exams across the full RRB ecosystem + NDA + JEE + NEET |
| Accessibility | Android APK deployable without cloud dependency — works with a local backend on a shared community device |
| Human oversight | The NAGA/Dabbu architecture ensures a human mentor reviews every AI-generated piece of content before students see it |
| Cost | Completely free. The only required API key is an OpenRouter free-tier key |

A student in a village with a ₹8,000 Android phone can now get the same adaptive, personalised exam preparation as a student in a city paying ₹20,000/month for coaching.

## Demo

**Video demo:** [https://youtu.be/AkeJM2bScQU](https://youtu.be/AkeJM2bScQU)

**GitHub repository:** [https://github.com/RKPrasada/GURUKULAI](https://github.com/RKPrasada/GURUKULAI)

**Android APK (sideload):** [GitHub Releases v1.0-kaggle](https://github.com/RKPrasada/GURUKULAI/releases/tag/v1.0-kaggle)

**Production backend:** `https://gurukulai-backend-242694625313.us-central1.run.app` (Google Cloud Run, free tier)

### Demo Script

| Scene | What to show |
|---|---|
| 1. Onboarding | Register → select RRB NTPC → 100Q diagnostic → topic-level weakness map |
| 2. Study loop | Ask about Number System → bilingual study notes → MCQ practice → wrong answer → Hindi explanation |
| 3. Study plan | Study Plan page → Dabbu-generated 13-week calendar → click day → 5-slot timeline |
| 4. Progress | Streak heatmap → SM-2 cards → Ask Dabbu → NAGA approves intervention |
| 5. Mock test | Start mock → timed exam UI → flag a question → review flagged before submit → result card |
| 6. NAGA | Post question → login as NAGA → answer it → back to student, notification appears |
| 7. Security | Phone number in input → amber guardrail card → SHA-256 audit log entry |


## What's Next

- Hindi-language diagnostic questions via LLM (currently English-only questions, Hindi explanations)
- Voice input for rural users with low literacy
- Offline question bank for areas with intermittent connectivity
- RRB CBT-2 discipline-specific mock tests (JE Technical, ALP Part B trade qualifying)
- Release-sign the APK with a proper keystore for Play Store distribution
- Email notifications via Gmail for class schedule and meeting confirmations